## **Double-pendulum model by PySINDy**

## **I. Setup and Preprocessing**

In [1]:
import os
import sys
sys.path.append("../utilities")

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score, mean_squared_error

import pysindy as ps

MY_DATA_TYPE = 'float32'
SEED = 42
np.random.seed(SEED)

opt_folder = 'output'
os.makedirs(opt_folder, exist_ok=True)

print(f"PySINDy version: {ps.__version__}")
print("Setup complete!")

PySINDy version: 2.0.0
Setup complete!


### **1.1 Load Data**

In [2]:
def load_double_pendulum_data(mat_path, downsample=1):
    """Load double pendulum data from .mat file."""
    data = loadmat(mat_path)
    theta1 = data['Theta1'][::downsample]
    theta2 = data['Theta2'][::downsample]
    d_theta1 = data['dTheta1'][::downsample]
    d_theta2 = data['dTheta2'][::downsample]
    
    my_data = np.concatenate([theta1, theta2, d_theta1, d_theta2], axis=1)
    d_theta = np.concatenate([d_theta1, d_theta2], axis=1)
    
    return my_data, d_theta

# Load training data
data_1, d_theta_1 = load_double_pendulum_data('DoublePendulum_Data/DoubleDataFreeSwing_1_Dt_0_001.mat', downsample=20)
data_2, d_theta_2 = load_double_pendulum_data('DoublePendulum_Data/DoubleDataFreeSwing_2_Dt_0_001.mat', downsample=20)

my_data = np.concatenate([data_1, data_2], axis=0)
dt = 0.001 * 20  # Time step

# Calculate second derivatives using Savitzky-Golay filter
window_length = 7
polyorder = 2
dd_theta_1 = savgol_filter(d_theta_1, window_length, polyorder, delta=dt, deriv=1, axis=0)
dd_theta_2 = savgol_filter(d_theta_2, window_length, polyorder, delta=dt, deriv=1, axis=0)

derivative_data_1 = np.concatenate([d_theta_1, dd_theta_1], axis=1)
derivative_data_2 = np.concatenate([d_theta_2, dd_theta_2], axis=1)
derivative_data = np.concatenate([derivative_data_1, derivative_data_2], axis=0)

print(f"Data shape: {my_data.shape}")
print(f"Derivative data shape: {derivative_data.shape}")

Data shape: (17669, 4)
Derivative data shape: (17669, 4)


### **1.2 Build Candidate Function Library**

In [3]:
poly_library = ps.PolynomialLibrary(degree=1, include_bias=True)

fourier_library = ps.FourierLibrary(n_frequencies=2)  # n_frequencies=2 gives sin(x), cos(x), sin(2x), cos(2x)

library_1st = poly_library + fourier_library

# create 2nd order by tensor product of two 1st order
combined_library = ps.GeneralizedLibrary(
    [library_1st, library_1st],  
    tensor_array=[[1,1]]
)

# create 3rd order by tensor product of 2nd order with 1st order
combined_library = ps.GeneralizedLibrary(
    [library_1st, combined_library],  
    tensor_array=[[1,1]]
)

## **II. Train SINDy Model**

In [4]:
# Create PySINDy model with STLSQ optimizer
model = ps.SINDy(
    feature_library=combined_library,
    optimizer=ps.STLSQ(threshold=0.005, alpha=0.0, max_iter=20)
)

# Fit the model
print("Training SINDy model...")
model.fit(my_data, t=dt, x_dot=derivative_data)

print("\nModel training complete!")
print("\nIdentified equations:")
model.print()

Training SINDy model...


/opt/anaconda3/envs/sreinet/lib/python3.10/site-packages/pysindy/optimizers/stlsq.py:269: ConvergenceWarning: STLSQ did not converge after {self.max_iter} iterations.
  warnings.warn(



Model training complete!

Identified equations:
(x0)' = -0.008 x2 cos(2 x2) + 0.106 1 x2 + 0.106 1 x2 + 0.106 1 1 x2 + 0.106 1 x2 1 + -0.008 1 x2 cos(2 x2) + 0.106 x2 1 + 0.106 x2 1 1 + 0.057 x2 sin(1 x2) sin(1 x2) + 0.049 x2 cos(1 x2) cos(1 x2) + 0.040 x2 sin(1 x3) sin(1 x3) + 0.026 x2 cos(2 x3) 1 + 0.035 sin(1 x0) x2 sin(1 x0) + 0.035 sin(1 x0) sin(1 x0) x2 + 0.057 sin(1 x2) x2 sin(1 x2) + 0.049 cos(1 x2) x2 cos(1 x2) + 0.049 cos(1 x2) cos(1 x2) x2 + 0.040 sin(1 x3) x2 sin(1 x3) + 0.040 sin(1 x3) sin(1 x3) x2 + 0.066 cos(1 x3) x2 cos(1 x3) + 0.035 cos(2 x0) x2 + 0.053 sin(2 x1) sin(2 x1) x2 + 0.053 cos(2 x1) x2 cos(2 x1) + 0.053 sin(2 x2) sin(2 x2) x2 + 0.053 cos(2 x2) cos(2 x2) x2
(x1)' = 0.200 x3 + 0.200 1 x3 + 0.200 1 1 x3 + 0.200 1 x3 1 + 0.100 x3 cos(2 x0) cos(2 x0) + 0.100 sin(1 x0) sin(1 x0) x3 + 0.100 cos(1 x0) cos(1 x0) x3 + 0.100 sin(2 x0) x3 sin(2 x0)
(x2)' = -51144.296 1 + 59107.530 x0 + -48312.215 x1 + 111840.937 x2 + -171815.849 x3 + 43893.045 sin(1 x0) + -64564.429 co

### **Model Complexity**

In [ ]:
# Get coefficient matrix
coefficients = model.coefficients()

# Calculate sparsity
total_coefficients = coefficients.size
nonzero_coefficients = np.count_nonzero(coefficients)
sparsity = (1 - nonzero_coefficients / total_coefficients) * 100

print(f"Coefficient matrix shape: {coefficients.shape}")
print(f"Total coefficients: {total_coefficients}")
print(f"Nonzero coefficients: {nonzero_coefficients}")
print(f"Sparsity: {sparsity:.1f}%")

## **III. Validation and Testing**

### **3.1 Load Validation Data and Simulate**

In [ ]:
# Load validation data
val_data, val_d_theta = load_double_pendulum_data(
    'DoublePendulum_Data/DoubleDataFreeSwing_3_Dt_0_001.mat', 
    downsample=20
)

# Simulation parameters
t_test = np.arange(0, 0.4, 0.02)
initial_condition = val_data[0]

# Simulate 
x_sim = model.simulate(initial_condition, t_test)

print(f"Simulation complete! Shape: {x_sim.shape}")

### **3.2 Plot Results - Positions**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

for i in range(2):
    axes[i].plot(t_test, x_sim[:, i], label='PySINDy', linewidth=2)
    axes[i].plot(t_test[:len(val_data)], val_data[:len(t_test), i], '--', label='True', linewidth=2)
    axes[i].set_ylabel(f'θ{i+1}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)')
plt.suptitle('Double Pendulum - Angular Positions (PySINDy)')
plt.tight_layout()
plt.show()

### **3.3 Plot Results - Velocities**

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6))

for i in range(2):
    axes[i].plot(t_test, x_sim[:, i+2], label='PySINDy', linewidth=2)
    axes[i].plot(t_test[:len(val_data)], val_data[:len(t_test), i+2], '--', label='True', linewidth=2)
    axes[i].set_ylabel(f'dθ{i+1}/dt')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (s)')
plt.suptitle('Double Pendulum - Angular Velocities (PySINDy)')
plt.tight_layout()
plt.show()

### **3.4 Model Score**

In [ ]:
# Calculate model score on training data
train_score = model.score(my_data, t=dt, x_dot=derivative_data)
print(f"Model score on training data: {train_score:.6f}")